## Week 1 primer: the language you need for the live coding

Computational Macro (HWS 2026), University of Mannheim

The course solves models in class, in notebooks that we fill in together. This
primer introduces the language constructs those notebooks use, and nothing
else, using the growth model of session 1 as the running example.

**Before you start.** Open this notebook in VS Code and pick the Julia kernel in the top right
corner. The kernel must run in the course environment of the `Julia` folder,
which pins the package versions: `Julia/README.md` explains how to activate it
(`] activate .` then `instantiate` in the Julia REPL started in that folder) and
how to register the kernel with `using IJulia`. If `using Plots` in the first
cell fails, the environment is not active.

**The model** (session 1, "The same example by dynamic programming"):

$$
v(k) = \max_{0 \leq k' \leq k^\alpha} \bigl\{ \ln(k^\alpha - k') + \beta \, v(k') \bigr\},
\qquad \alpha = 0.3, \quad \beta = 0.96 .
$$

**What we derived** (guess and verify, and "VFI by hand"):

$$
v(k) = E + F \ln k, \quad F = \frac{\alpha}{1 - \alpha\beta}, \qquad
k' = \alpha\beta \, k^\alpha, \qquad
F_{n+1} = \alpha + \alpha\beta \, F_n \ \xrightarrow{n \to \infty} \ F .
$$

Five short parts: arrays, containers, functions, loops, plots. Run every cell
with Shift+Enter and change things to see what happens.

In [ ]:
using Plots
using Printf

### 1. Arrays and elementwise arithmetic

A function evaluated on a grid is a vector: $N$ numbers, one per grid point. The
first object we need is the grid itself, $\mathcal{K} = \{k_1, \ldots, k_N\}$, then
output $k_i^\alpha$ and the closed form policy $\alpha\beta k_i^\alpha$ at every
point. In Julia, `range` builds evenly spaced points and `collect` turns them into a
vector. Arithmetic on a whole vector needs a **dot** before the operator, `.^`,
`.*`, `.-`, which means "apply elementwise". Functions are applied elementwise
with a dot after the name, `log.(k)`.

In [ ]:
α, β = 0.3, 0.96
kstar = (α * β)^(1 / (1 - α))                # steady state of the closed form policy
k = collect(range(0.1 * kstar, 2.0 * kstar, length = 10))   # a small grid to look at
output = k .^ α                              # k_i^α at every grid point, the dot makes it elementwise
policy = α * β .* output                     # αβ k_i^α, the closed form policy
[k output policy]                            # three columns side by side

A second way to place points: equidistant in $\ln k$ rather than in $k$. Take
the logs of the bounds, space them evenly, and map back with the exponential.
The points are denser at low $k$.

In [ ]:
k_log = exp.(range(log(0.1 * kstar), log(2.0 * kstar), length = 10))
println("equidistant, spacing at the start and at the end: ", round(k[2] - k[1], digits = 4), "  ", round(k[end] - k[end-1], digits = 4))
println("log spaced,  spacing at the start and at the end: ", round(k_log[2] - k_log[1], digits = 4), "  ", round(k_log[end] - k_log[end-1], digits = 4))

### 2. Containers for parameters and grids

Parameters live in a container with named fields, one for the economics and one
for the numerics, so that a change to the grid can never touch $\alpha$ or
$\beta$ by accident. `Base.@kwdef struct` defines such a container with default values. It is
created with keyword arguments, and any field can be overridden at creation,
`NumericalParameters(nk = 30)`. Fields are read with a dot, `par.α`. A named
tuple, `(k = ...,)`, is the lighter cousin, used for the grid. The comma before
the closing bracket matters: it is what makes a one-field tuple a tuple.

In [ ]:
Base.@kwdef struct EconomicParameters
    α::Float64 = 0.3          # capital share
    β::Float64 = 0.96         # discount factor
end

Base.@kwdef struct NumericalParameters
    nk::Int = 50              # number of grid points
    crit::Float64 = 1e-8      # convergence tolerance
    maxiter::Int = 1000       # iteration cap
end

par = EconomicParameters()
mpar = NumericalParameters(nk = 40)          # override one default, keep the others
gri = (k = collect(range(0.1 * kstar, 2.0 * kstar, length = mpar.nk)),)   # named tuple, note the comma
println(par)
println(mpar)
println("grid with ", length(gri.k), " points, read with gri.k")

### 3. Functions

A function takes arguments and returns a value. Julia has two ways to write one. The short form, `name(x) = expression`, for
a function that fits on one line. The long form, `function name(x) ... return
... end`, for anything longer. Both are called the same way, `name(2.0)`.

Three functions from session 1: the utility function $u(c) = \ln c$, the closed
form value function $v(k) = E + F \ln k$, and the policy $k' = \alpha\beta k^\alpha$.
The constants $E$ and $F$ are computed once from the parameters and used inside
the functions.

In [ ]:
util(c) = log(c)                             # short form: one line

F_star = par.α / (1 - par.α * par.β)
E_star = (log(1 - par.α * par.β) + par.β * F_star * log(par.α * par.β)) / (1 - par.β)

function v_true(k)                           # long form: several lines, explicit return
    return E_star + F_star * log(k)
end

policy_true(k) = par.α * par.β .* k .^ par.α # written with dots, so it accepts a vector

@printf("u(1) = %.3f, u(2) = %.3f, v(k*) = %.4f, k'(k*) = %.4f\n", util(1.0), util(2.0), v_true(kstar), policy_true(kstar))

Functions applied to a vector give a vector, so evaluating the closed form on
the whole grid is one line.


In [ ]:
v_grid = v_true.(gri.k)                      # the dot applies v_true to every grid point
policy_grid = policy_true(gri.k)             # elementwise operators inside, no dot needed
[gri.k[1:5] v_grid[1:5] policy_grid[1:5]]

### 4. Loops

A `while` loop repeats its body as long as a condition holds. The condition can
be anything: a counter below a limit, an input not yet received, or, the case we
care about, a change between two iterates that is still larger than a tolerance.
That last use is how every solution method of this course is written: iterate
until the answer stops moving, with an iteration cap as a safety net.

The recursion $F_{n+1} = \alpha + \alpha\beta F_n$ from "VFI by hand" is the
smallest example. It should converge to $F = \alpha / (1 - \alpha\beta)$ from any
$F_0$. Two Julia details. Variables assigned inside a `while` loop at the top level
of a notebook need a `global` declaration at the start of the loop body, otherwise
Julia treats them as local to the loop. And `push!` appends to a vector, the
exclamation mark being the convention for functions that modify their argument.

In [ ]:
F = 0.0                                      # the guess F_0 = 0, as on the slide
dist = [Inf]                                 # distances, one per iteration
n = 0
while dist[end] > mpar.crit && n < mpar.maxiter
    global F, n
    F_new = par.α + par.α * par.β * F        # the recursion from "VFI by hand"
    push!(dist, abs(F_new - F))              # record how much F moved
    F = F_new
    n += 1
end
@printf("after %d iterations: F_n = %.8f, the closed form F = %.8f\n", n, F, F_star)

The contraction mapping theorem says the distance shrinks by a fixed factor per
step, here $\alpha\beta = 0.288$. Check it on the recorded distances: the ratio of
successive distances should settle on that number. A `for` loop, which runs a
fixed number of times, prints the first few.

In [ ]:
for i in 3:7
    @printf("iteration %d: distance %.2e, ratio to the previous one %.4f\n", i - 1, dist[i], dist[i] / dist[i-1])
end
println("αβ = ", par.α * par.β)

### 5. Plots

Three plots: the closed form value function on the grid, the closed form policy
against the 45 degree line with the steady state $k^*$ marked, and the
convergence of the recursion on a log scale, where geometric convergence is a
straight line. With Plots.jl, `plot` starts a figure and `plot!` adds to it, `scatter!` adds
points. Legend labels are keyword arguments, `yaxis = :log10` switches the axis,
`hline!` draws a horizontal line, and `layout` puts several plots side by side.

In [ ]:
p1 = plot(gri.k, v_grid, linewidth = 2, label = "v(k) = E + F ln k", legend = :bottomright,
          xlabel = "capital k", title = "Closed form value function")

p2 = plot(gri.k, policy_grid, linewidth = 2, label = "k' = αβ k^α", legend = :topleft,
          xlabel = "capital today k", ylabel = "capital tomorrow k'", title = "Closed form policy")
plot!(p2, gri.k, gri.k, color = :gray, linestyle = :dash, label = "45 degree line")
scatter!(p2, [kstar], [kstar], color = :black, label = "steady state k*")

p3 = plot(1:n, dist[2:end], yaxis = :log10, linewidth = 2, label = "|F_n - F_(n-1)|",
          xlabel = "iteration n", title = "Convergence of the recursion")
hline!(p3, [mpar.crit], color = :gray, linestyle = :dot, label = "tolerance")

plot(p1, p2, p3, layout = (1, 3), size = (1400, 400), margin = 5Plots.mm)

### Exercises

1. In part 4, start from $F_0 = 40$ and from $F_0 = -5$. Does the limit change? Does the number of iterations?
2. Define CRRA utility $u(c) = (c^{1-\gamma} - 1)/(1-\gamma)$ as a second function with $\gamma = 2$ and plot both utility functions on a grid of consumption levels between $0.1$ and $2$. Where do they differ most?
3. Change the grid in part 1 to 20 points and to 500 points and rerun part 5. What changes in the plots, and what does not?